<a href="https://colab.research.google.com/github/claramanolache/DeepLearning/blob/main/Stock_Market.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets
!pip install legacy-cgi

from practice notebook (added opendatasets for kaggle import)

In [2]:
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils import data

import opendatasets as od
import pandas as pds

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

#Importing Dataset

In [4]:
od.download(
    "https://www.kaggle.com/datasets/zahrashahzahi/stock-market-dataset/data")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: claramanolache
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/zahrashahzahi/stock-market-dataset


100%|██████████| 2.08M/2.08M [00:00<00:00, 155MB/s]

In [5]:
import os

folder = "stock-market-dataset"

print(os.listdir(folder))

['Market.csv']


In [6]:
file =('stock-market-dataset/Market.csv')
df = pds.read_csv(file)

In [7]:
df.shape
df.head()

,Index,Date,Open,High,Low,Close,Adj Close,Volume
0,NYA,12/31/1965,528.690002,528.690002,528.690002,528.690002,528.690002,0.0
1,NYA,1/3/1966,527.210022,527.210022,527.210022,527.210022,527.210022,0.0
2,NYA,1/4/1966,527.840027,527.840027,527.840027,527.840027,527.840027,0.0
3,NYA,1/5/1966,531.119995,531.119995,531.119995,531.119995,531.119995,0.0
4,NYA,1/6/1966,532.070007,532.070007,532.070007,532.070007,532.070007,0.0


In [13]:
print(df.shape)

(112457, 8)


#Dataset Visuals

In [8]:
df['Index'].unique()

array(['NYA', 'IXIC', 'HSI', '000001.SS', 'GSPTSE', '399001.SZ', 'NSEI',
       'GDAXI', 'KS11', 'SSMI', 'TWII', 'J203.JO', 'N225', 'N100'],
      dtype=object)

In [9]:
count = (df['Index'] == 'NYA').sum()
print(count)

13948


#Redoing split
might have to only be one index

In [44]:
df['Date'] = pds.to_datetime(df['Date'])
df = df.set_index('Date')

In [61]:
df_nya = df[df['Index'] == 'NYA'].copy()
df_nya = df_nya.sort_values('Date')
df_nya = df_nya.drop(columns=['Index'])

print(df_nya.shape)
df_nya.head()

(13948, 6)


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1965-12-31,528.690002,528.690002,528.690002,528.690002,528.690002,0.0
1966-01-03,527.210022,527.210022,527.210022,527.210022,527.210022,0.0
1966-01-04,527.840027,527.840027,527.840027,527.840027,527.840027,0.0
1966-01-05,531.119995,531.119995,531.119995,531.119995,531.119995,0.0
1966-01-06,532.070007,532.070007,532.070007,532.070007,532.070007,0.0


In [82]:
# Filter to one index and keep its observations in chronological order.
df_nya = df.loc[df['Index'] == 'NYA'].copy()
df_nya = df_nya.sort_index().drop(columns=['Index'])

# All model features must be numeric. Remove invalid market observations
# BEFORE splitting or constructing windows.
df_nya = df_nya.apply(pds.to_numeric, errors='coerce')
df_nya = df_nya.replace([np.inf, -np.inf], np.nan)
print('Missing values by column (before cleaning):')
print(df_nya.isna().sum())
old_count = len(df_nya)
df_nya = df_nya.dropna()
print(f'Removed {old_count - len(df_nya)} incomplete rows; {len(df_nya)} NYA rows remain.')
print(df_nya.shape)
df_nya.head()

Missing values by column (before cleaning):
Open          1
High          2
Low           3
Close         4
Adj Close    10
Volume        1
dtype: int64
Removed 16 incomplete rows; 13932 NYA rows remain.
(13932, 6)


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1965-12-31,528.690002,528.690002,528.690002,528.690002,528.690002,0.0
1966-01-03,527.210022,527.210022,527.210022,527.210022,527.210022,0.0
1966-01-04,527.840027,527.840027,527.840027,527.840027,527.840027,0.0
1966-01-05,531.119995,531.119995,531.119995,531.119995,531.119995,0.0
1966-01-06,532.070007,532.070007,532.070007,532.070007,532.070007,0.0


In [83]:
df_train = df_nya[:11000]
df_val = df_nya[11000:12554]
df_test = df_nya[12554:]

##Normilizing

In [100]:

train_mean = df_train.mean()
train_std = df_train.std()


In [101]:
# Fit normalization statistics using training observations ONLY.
train_mean = df_train.mean()
train_std = df_train.std()
print('Training standard deviations:')
print(train_std)
zero_std_cols = train_std.index[(train_std == 0) | (~np.isfinite(train_std))].tolist()
if zero_std_cols:
    print('Warning: constant/invalid-std columns:', zero_std_cols)
    # A constant training feature becomes zero after scaling.
    train_std.loc[zero_std_cols] = 1.0

df_train_norm = (df_train - train_mean) / train_std
df_val_norm = (df_val - train_mean) / train_std
df_test_norm = (df_test - train_mean) / train_std

for name, frame in [('train', df_train_norm), ('validation', df_val_norm), ('test', df_test_norm)]:
    print(f'{name} normalization finite:', np.isfinite(frame.to_numpy()).all())
    if not np.isfinite(frame.to_numpy()).all():
        print(frame.columns[~np.isfinite(frame.to_numpy()).all(axis=0)].tolist())
        raise ValueError(f'Non-finite normalized values in {name}; inspect the listed columns.')

Training standard deviations:
Open         2.714328e+03
High         2.726737e+03
Low          2.700984e+03
Close        2.714460e+03
Adj Close    2.714555e+03
Volume       1.282879e+09
dtype: float64
train normalization finite: True
validation normalization finite: True
test normalization finite: True


In [102]:
X_train = df_train_norm.drop(columns=['Close', 'Adj Close'])
y_train = df_train_norm['Close']

X_val = df_val_norm.drop(columns=['Close', 'Adj Close'])
y_val = df_val_norm['Close']

X_test = df_test_norm.drop(columns=['Close', 'Adj Close'])
y_test = df_test_norm['Close']

In [103]:
X_train.shape

(11000, 4)

In [66]:
input_features = 4

# Hyperparameters

experimenting with window size

In [104]:
window_size_1 = 15
window_size_2 = 30
window_size_3 = 100

In [105]:
n_features = X_train.shape[1]

def window_dataset(X, window_size, n_features):
    X_win = np.zeros((
        len(X) - window_size,
        window_size,
        n_features
    ))

    for i in range(len(X) - window_size):
        X_win[i] = X[i:i + window_size]

    X_win = torch.tensor(X_win).float()

    return X_win.reshape(
        -1, window_size, n_features
    )

In [106]:
model = nn.Sequential(
    nn.RNN(
        input_size=X_train.shape[1],
        hidden_size=32,
        num_layers=1,
        batch_first=True
    ),
    nn.Linear(32, 1)
)

TODO: make more model sizes?

In [107]:
n_epochs = 10
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()
patience = 5

#Setting up models

code from the practice notebook

In [108]:
X_train_win = window_dataset(
    X_train.to_numpy(), window_size_1, n_features
)

X_val_win = window_dataset(
    X_val.to_numpy(), window_size_1, n_features
)

X_test_win = window_dataset(
    X_test.to_numpy(), window_size_1, n_features
)

y_train_win = torch.tensor(
    y_train.iloc[window_size_1:].values
).float().reshape(-1, 1)

y_val_win = torch.tensor(
    y_val.iloc[window_size_1:].values
).float().reshape(-1, 1)

y_test_win = torch.tensor(
    y_test.iloc[window_size_1:].values
).float().reshape(-1, 1)

dimension check

In [109]:
print("Number of features:", n_features)
print("X_train:", X_train.shape)
print("X_train_win:", X_train_win.shape)
print("y_train_win:", y_train_win.shape)

Number of features: 4
X_train: (11000, 4)
X_train_win: torch.Size([10985, 15, 4])
y_train_win: torch.Size([10985, 1])


In [110]:
def create_data_loader(X, y, batch_size=32, shuffle=True):
    return data.DataLoader(
        data.TensorDataset(
            torch.FloatTensor(X),
            torch.FloatTensor(y)
        ),
        batch_size=batch_size,
        shuffle=shuffle
    )

In [116]:
train_loader = create_data_loader(X_train_win, y_train_win)
val_loader = create_data_loader(
    X_val_win, y_val_win, shuffle=False
)
test_loader = create_data_loader(X_test_win, y_test_win, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)

(344, 49, 43)

In [112]:
def get_next_prediction(
    model: nn.Sequential, inputs: torch.Tensor
) -> torch.Tensor:
    outputs, _ = model[0](inputs)
    last_time_step = outputs[:, -1, :]
    return model[1](last_time_step)

In [120]:
avg_costs = []
best_cost = float("inf")
best_params = None

model.to(device)
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      optimizer.zero_grad()

      y_pred = get_next_prediction(model, X_batch)
      loss = criterion(y_pred, y_batch)

      if not torch.isfinite(loss):
          raise ValueError("Non-finite training loss")

      loss.backward()

      torch.nn.utils.clip_grad_norm_(
          model.parameters(), max_norm=1.0
      )

      optimizer.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = get_next_prediction(model, X_batch)

            loss = criterion(y_pred, y_batch)
            val_loss += loss.item()
    avg_costs.append(val_loss / len(val_loader))

    print(f"Epoch {epoch:3d} | Validation Cost: {avg_costs[-1]:.4f} | Best: {best_cost:.4f}")
    if avg_costs[-1] < best_cost:
        best_cost = avg_costs[-1]
        best_params = copy.deepcopy(model.state_dict())
    elif len(avg_costs) > patience and min(avg_costs[-patience:]) > best_cost:
        print("Early Stopping Triggered")
        break

Epoch   0 | Validation Cost: 0.0021 | Best: inf
Epoch   1 | Validation Cost: 0.0027 | Best: 0.0021
Epoch   2 | Validation Cost: 0.0029 | Best: 0.0021
Epoch   3 | Validation Cost: 0.0020 | Best: 0.0021
Epoch   4 | Validation Cost: 0.0037 | Best: 0.0020
Epoch   5 | Validation Cost: 0.0024 | Best: 0.0020
Epoch   6 | Validation Cost: 0.0048 | Best: 0.0020
Epoch   7 | Validation Cost: 0.0051 | Best: 0.0020
Epoch   8 | Validation Cost: 0.0026 | Best: 0.0020
Early Stopping Triggered


#eval

In [121]:
if best_params is not None:
    model.load_state_dict(best_params)

model.eval()

Sequential(
  (0): RNN(4, 32, batch_first=True)
  (1): Linear(in_features=32, out_features=1, bias=True)
)

In [122]:
model.eval()
predictions, actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        y_pred = get_next_prediction(model, X_batch.to(device)).cpu().numpy()
        predictions.append(y_pred)
        actuals.append(y_batch.numpy())
pred_scaled = np.concatenate(predictions).reshape(-1)
actual_scaled = np.concatenate(actuals).reshape(-1)
close_mean, close_std = train_mean['Close'], train_std['Close']
pred_price = pred_scaled * close_std + close_mean
actual_price = actual_scaled * close_std + close_mean
mae = np.mean(np.abs(pred_price - actual_price))
rmse = np.sqrt(np.mean((pred_price - actual_price) ** 2))
print(f'Test MAE: {mae:.2f}; Test RMSE: {rmse:.2f}')


Test MAE: 700.88; Test RMSE: 910.57


In [124]:
mae = np.mean(np.abs(pred_price - actual_price))
rmse = np.sqrt(np.mean((pred_price - actual_price) ** 2))
mae, rmse

(np.float64(700.8798246313652), np.float64(910.568520943498))

#Graveyard of code


In [117]:
"""
model.to(device)
model.train()
for epoch in range(n_epochs):
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        y_pred = get_next_prediction(model, X_batch)
        loss = criterion(y_pred, y_batch)

        if not torch.isfinite(loss):
            raise ValueError("Non-finite training loss")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

In [97]:
'''
X_train_win = window_dataset(X_train.to_numpy(), window_size_1, n_features)
X_val_win = window_dataset(X_val.to_numpy(), window_size_1, n_features)
X_test_win = window_dataset(X_test.to_numpy(), window_size_1, n_features)

y_train_win = torch.tensor(y_train.iloc[window_size_1:].to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_val_win = torch.tensor(y_val.iloc[window_size_1:].to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_test_win = torch.tensor(y_test.iloc[window_size_1:].to_numpy(), dtype=torch.float32).reshape(-1, 1)

for name, X_win, y_win in [('train', X_train_win, y_train_win),
                           ('validation', X_val_win, y_val_win),
                           ('test', X_test_win, y_test_win)]:
    assert len(X_win) == len(y_win) and len(X_win) > 0, f'{name}: mismatched/empty windows'
    print(f'{name}: X={tuple(X_win.shape)}, y={tuple(y_win.shape)}')
    print(f'{name}: finite inputs={torch.isfinite(X_win).all().item()}, finite targets={torch.isfinite(y_win).all().item()}')
    assert torch.isfinite(X_win).all() and torch.isfinite(y_win).all(), f'{name}: NaN or Inf in windows'


train: X=(10985, 15, 4), y=(10985, 1)
train: finite inputs=True, finite targets=True
validation: X=(1539, 15, 4), y=(1539, 1)
validation: finite inputs=True, finite targets=True
test: X=(1363, 15, 4), y=(1363, 1)
test: finite inputs=True, finite targets=True
